# FXS / 03h — Closing at the "optimal" holding time (look-ahead, for illustration only)

Iván (2026-11-11): what would the P&L be if each structure closed at its best holding time from 03g? **This uses look-ahead bias**: the exit time is chosen from the same 2016–2026 data it is tested on, so these results are an upper bound, not an expectation.

Two definitions of "optimal", per structure (entry unchanged, no stop, no take-profit):
- `OPT_ALL`: the holding time with the highest mean net pips over 2016–2026.
- `OPT_AVG`: the average of the best holding times of 2016–2020, 2021–2025 and 2026, rounded to 5 minutes.

Compared with `BASE` (the frozen exit). Costs: Dukascopy spread × `SPREAD_FACTOR`, `COMMISSION_PER_LOT` per lot. Account: $6,000, compounding, L × equity per structure, all four structures, and without EURUSD-01.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

COMMISSION_PER_LOT = 3.0
SPREAD_FACTOR = 0.9
STEP = 5
IDS = ["EURUSD-01", "EURUSD-02", "USDJPY-01", "USDJPY-02"]
FROZEN = {"EURUSD-01": ("UTC", 75, 195, 1), "EURUSD-02": ("NY", 660, 900, -1),
          "USDJPY-01": ("UTC", 55, 115, -1), "USDJPY-02": ("UTC", 25, 55, 1)}
WINDOWS = ((dt.date(2016, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
           (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))
PERIODS = {"2016-2020 (in sample)": (2016, 2020), "2021-2025": (2021, 2025), "2026": (2026, 2026)}

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs03h_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("03h_synthetic" if SYNTHETIC else "03h"); OUT.mkdir(exist_ok=True)
E0 = 6_000.0
LEVERAGES = [1, 2, 3, 5, 10]
pd.set_option("display.width", 220, "display.max_columns", 60)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(("EURUSD", "USDJPY")):
        F.write_synthetic_candles(base, p, dt.date(2015, 12, 28), dt.date(2026, 11, 6), seed=21 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 6), price=150.0 if p.endswith("JPY") else 1.10)
    cand = pd.DataFrame([{"id": k, "pair": k[:6], "clock": c, "start": s, "end": e, "direction": d} for k, (c, s, e, d) in FROZEN.items()])
else:
    cand = F.load_frozen_candidates(BRANCH).query("id in @IDS").reset_index(drop=True)
for _, r in cand.iterrows():
    assert (r["clock"], int(r["start"]), int(r["end"]), int(r["direction"])) == FROZEN[r["id"]], f"{r['id']} is not the frozen rule"
cand = cand.set_index("id").loc[IDS].reset_index()

## 1. Net pips for every holding time, every day

In [ ]:
rows = []
for (pair, clock), grp in cand.groupby(["pair", "clock"]):
    for a, b, stage in WINDOWS:
        p = F.load_panels([pair], a, b, stage, cfg, clocks=(clock,))[(pair, clock)]
        mid = 0.5 * (p.bid + p.ask)
        half = 0.5 * (p.ask - p.bid) * SPREAD_FACTOR
        bid, ask = mid - half, mid + half
        pip = F.pip_size(pair)
        years = p.years
        for _, r in grp.iterrows():
            s, base, d = int(r["start"]), int(r["end"]) - int(r["start"]), int(r["direction"])
            holds = np.arange(STEP, int(1.5 * base) + 1, STEP)
            holds = holds[s + holds <= F.SLOTS]
            entry = ask[:, s] if d > 0 else bid[:, s]
            comm_pips = F.bps_to_pips(F.commission_bps_per_lot(pair, COMMISSION_PER_LOT), mid[:, s], pair)
            for h in holds:
                exit_ = bid[:, s + h] if d > 0 else ask[:, s + h]
                net = (exit_ - entry) * d / pip - comm_pips
                rows.append(pd.DataFrame({"structure": r["id"], "hold": int(h), "base": base, "year": years, "net_pips": net,
                                          "mid_pips": (mid[:, s + h] - mid[:, s]) * d / pip, "date": p.dates, "price": mid[:, s]}))
H = pd.concat(rows, ignore_index=True)
H["period"] = None
for name, (a, b) in PERIODS.items():
    H.loc[H["year"].between(a, b), "period"] = name
print(H.groupby("structure")["hold"].agg(["min", "max", "nunique"]))

## 2. The chosen holding times

In [ ]:
S = H.groupby(["structure", "period", "hold"])["net_pips"].mean().rename("net").reset_index()
S_all = H.groupby(["structure", "hold"])["net_pips"].mean().rename("net").reset_index()
choice = []
for sid in IDS:
    base = int(H.loc[H["structure"] == sid, "base"].iloc[0])
    a = S_all[S_all["structure"] == sid].set_index("hold")["net"]
    per_best = {per: int(S[(S["structure"] == sid) & (S["period"] == per)].set_index("hold")["net"].idxmax()) for per in PERIODS}
    avg = int(STEP * round(np.mean(list(per_best.values())) / STEP))
    choice.append({"structure": sid, "BASE": base, "OPT_ALL": int(a.idxmax()), "OPT_AVG": avg, **{f"best {k}": v for k, v in per_best.items()}})
choice = pd.DataFrame(choice).set_index("structure")
display(choice); choice.to_csv(OUT / "03h_holding_times.csv")

## 3. Net pips per trade at each exit, by period

In [ ]:
rows = []
for sid in IDS:
    for v in ("BASE", "OPT_ALL", "OPT_AVG"):
        h = int(choice.loc[sid, v])
        x = H[(H["structure"] == sid) & (H["hold"] == h)]
        r = {"structure": sid, "variant": v, "hold": h}
        for per in list(PERIODS) + ["all"]:
            y = x if per == "all" else x[x["period"] == per]
            r[per] = y["net_pips"].mean()
        r["win rate"] = (x["net_pips"] > 0).mean()
        rows.append(r)
tab = pd.DataFrame(rows).set_index(["structure", "variant"])
display(tab.round(2)); tab.to_csv(OUT / "03h_pips.csv")

## 4. The $6,000 compounding account

In [ ]:
def returns_bps(variant, ids):
    cols = {}
    for sid in ids:
        h = int(choice.loc[sid, variant])
        x = H[(H["structure"] == sid) & (H["hold"] == h)]
        pair = sid[:6]
        cols[sid] = pd.Series((x["net_pips"] * F.pip_size(pair) / x["price"] * 1e4).to_numpy(), index=pd.to_datetime(x["date"].to_numpy()))
    return pd.DataFrame(cols).sort_index().fillna(0.0)

acc_rows, curves, yearly = [], {}, {}
for set_name, ids in (("all four", IDS), ("without EURUSD-01", [i for i in IDS if i != "EURUSD-01"])):
    for v in ("BASE", "OPT_ALL", "OPT_AVG"):
        R = returns_bps(v, ids)
        for L in LEVERAGES:
            acc = F.simulate_account(R, pd.Series(float(L), index=ids), E0)
            st = F.account_stats(acc, E0)
            curves[(set_name, v, L)] = acc
            y = acc["equity"].resample("YE").last(); yr = y.pct_change(); yr.iloc[0] = y.iloc[0] / E0 - 1; yr.index = yr.index.year
            yearly[(set_name, v, f"{L}x")] = yr
            acc_rows.append({"set": set_name, "variant": v, "L each": L, "final_equity": round(st["final_equity"]), "cagr %": round(100 * st["cagr"], 1),
                             "max drawdown %": round(100 * st["max_drawdown"], 1), "worst year %": round(100 * st["worst_year"], 1)})
acc_tab = pd.DataFrame(acc_rows)
display(acc_tab); acc_tab.to_csv(OUT / "03h_account.csv", index=False)
yr_tab = 100 * pd.DataFrame(yearly)
display(yr_tab.xs("without EURUSD-01", axis=1, level=0).round(1)); yr_tab.to_csv(OUT / "03h_yearly_returns.csv")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
for ax, v in zip(axes, ("BASE", "OPT_ALL", "OPT_AVG")):
    for L in LEVERAGES:
        ax.plot(curves[("without EURUSD-01", v, L)]["equity"].clip(lower=1), label=f"{L}x each")
    ax.axhline(E0, color="grey", lw=0.6)
    for b_ in ("2021-01-01", "2026-01-01"):
        ax.axvline(pd.Timestamp(b_), color="black", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_title(f"{v} (without EURUSD-01)"); ax.legend()
axes[0].set_ylabel("equity USD (log)")
plt.tight_layout(); plt.savefig(OUT / "03h_equity.png", dpi=110); plt.show()

In [ ]:
summary = {"study": "FXS03h", "lookahead": True, "synthetic": SYNTHETIC, "holding_times": choice.reset_index().to_dict(orient="records"),
           "pips": tab.round(4).reset_index().to_dict(orient="records"), "account": acc_tab.to_dict(orient="records"),
           "note": "Exit times chosen on the same data they are tested on: an upper bound, not an expectation."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")